<a href="https://colab.research.google.com/github/WeegorMartins/customer-decisioning-lab/blob/main/notebooks/03_criteo_causal_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# 03 — Benchmark causal Criteo

Pergunta: qual pessoa apresenta maior efeito incremental do tratamento?

As variáveis f0...f11 são anônimas. Nenhuma interpretação bancária será criada.
```

# Parte A — benchmark Criteo

In [1]:
!pip -q install lightgbm

In [2]:
from pathlib import Path
import random
import time

import joblib
import numpy as np
import pandas as pd

from lightgbm import (
    LGBMClassifier,
    LGBMRegressor
)

from sklearn.model_selection import (
    train_test_split,
    KFold
)

from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("IMPORTAÇÕES PRONTAS")

IMPORTAÇÕES PRONTAS


In [3]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/customer-decisioning-lab"
)

CRITEO_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "criteo_sample_1500k.parquet"
)

print("Existe:", CRITEO_PATH.exists())
print(CRITEO_PATH)

Mounted at /content/drive
Existe: True
/content/drive/MyDrive/customer-decisioning-lab/data/processed/criteo_sample_1500k.parquet


In [4]:
criteo = pd.read_parquet(CRITEO_PATH)

FAST_MODE = True

if FAST_MODE:
    criteo = criteo.sample(
        n=min(500_000, len(criteo)),
        random_state=SEED
    ).reset_index(drop=True)

print("Formato usado:", criteo.shape)
display(criteo.head())

Formato usado: (500000, 16)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,25.600078,10.059654,8.214383,4.679882,10.280525,4.115453,-4.595460,4.833815,3.971858,13.190056,5.300375,-0.168679,0,0,0,0
1,22.634055,10.059654,8.214383,4.679882,10.280525,4.115453,-5.576414,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.670412,4.679882,10.280525,4.115453,0.294443,4.833815,3.934656,20.050937,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,8.427693,4.679882,10.280525,4.115453,0.294443,4.833815,3.927254,21.416100,5.300375,-0.168679,0,0,0,0
4,14.900920,10.059654,8.214383,2.293959,10.280525,4.115453,-5.116672,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0


# Parte B — definir X, T e Y

In [5]:
FEATURES = [f"f{i}" for i in range(12)]
TREATMENT = "treatment"
OUTCOME = "conversion"

X = criteo[FEATURES].copy()
T = criteo[TREATMENT].astype(int).copy()
Y = criteo[OUTCOME].astype(int).copy()

print("X:", X.shape)
print("Tratados:", T.mean())
print("Conversão:", Y.mean())

X: (500000, 12)
Tratados: 0.84974
Conversão: 0.002864


- `X`: como a pessoa era antes;
- `T`: recebeu ação ou não;
- `Y`: converteu ou não.

In [6]:
strata = (
    T.astype(str)
    + "_"
    + Y.astype(str)
)

train_idx, test_idx = train_test_split(
    np.arange(len(criteo)),
    test_size=0.30,
    random_state=SEED,
    stratify=strata
)

train_strata = strata.iloc[train_idx]

train_idx, valid_idx = train_test_split(
    train_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=train_strata
)

print("Treino:", len(train_idx))
print("Validação:", len(valid_idx))
print("Teste:", len(test_idx))

Treino: 280000
Validação: 70000
Teste: 150000


- treino: aprende;
- validação: ajuda a escolher;
- teste: prova final.

In [7]:
X_train = X.iloc[train_idx].reset_index(drop=True)
T_train = T.iloc[train_idx].reset_index(drop=True)
Y_train = Y.iloc[train_idx].reset_index(drop=True)

X_valid = X.iloc[valid_idx].reset_index(drop=True)
T_valid = T.iloc[valid_idx].reset_index(drop=True)
Y_valid = Y.iloc[valid_idx].reset_index(drop=True)

X_test = X.iloc[test_idx].reset_index(drop=True)
T_test = T.iloc[test_idx].reset_index(drop=True)
Y_test = Y.iloc[test_idx].reset_index(drop=True)

print("SEPARAÇÃO PRONTA")

SEPARAÇÃO PRONTA


# Parte C — efeito médio sem ML

In [8]:
test_observed = pd.DataFrame({
    "treatment": T_test,
    "conversion": Y_test
})

mean_table = (
    test_observed
    .groupby("treatment")
    .agg(
        customers=("conversion", "size"),
        conversion_rate=("conversion", "mean")
    )
)

display(mean_table)

,customers,conversion_rate
treatment,,
0,22539,0.001775
1,127461,0.003060


In [9]:
print("Índices existentes:", mean_table.index.tolist())

display(mean_table)

print("\nQuantidade por tratamento:")
display(
    test_observed["treatment"]
    .value_counts(dropna=False)
    .sort_index()
)

Índices existentes: [0, 1]


,customers,conversion_rate
treatment,,
0,22539,0.001775
1,127461,0.003060



Quantidade por tratamento:


,count
treatment,
0,22539
1,127461


In [10]:
treated_rate = mean_table.loc[
    1,
    "conversion_rate"
]

control_rate = mean_table.loc[
    0,
    "conversion_rate"
]

ate = treated_rate - control_rate

print("Taxa tratada:", treated_rate)
print("Taxa controle:", control_rate)
print("ATE:", ate)
print("ATE em pontos percentuais:", ate * 100)

Taxa tratada: 0.0030597594558335493
Taxa controle: 0.001774701628288744
ATE: 0.0012850578275448053
ATE em pontos percentuais: 0.12850578275448052


# Parte D — parâmetros comuns


In [11]:
MODEL_PARAMS = {
    "n_estimators": 250,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 100,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1
}

print(MODEL_PARAMS)

{'n_estimators': 250, 'learning_rate': 0.05, 'num_leaves': 31, 'min_child_samples': 100, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}


# Parte E — baseline de propensão de resposta

In [12]:
response_model = LGBMClassifier(
    **MODEL_PARAMS
)

response_model.fit(
    X_train,
    Y_train
)

response_score = response_model.predict_proba(
    X_test
)[:, 1]

response_auc = roc_auc_score(
    Y_test,
    response_score
)

print("AUC de resposta:", response_auc)

AUC de resposta: 0.9482754827404452


# Parte F — S-Learner

In [13]:
X_train_s = X_train.copy()
X_train_s["treatment"] = T_train.values

s_learner = LGBMClassifier(
    **MODEL_PARAMS
)

s_learner.fit(
    X_train_s,
    Y_train
)

print("S-LEARNER TREINADO")

S-LEARNER TREINADO


In [14]:
X_test_treated = X_test.copy()
X_test_treated["treatment"] = 1

X_test_control = X_test.copy()
X_test_control["treatment"] = 0

s_mu1 = s_learner.predict_proba(
    X_test_treated
)[:, 1]

s_mu0 = s_learner.predict_proba(
    X_test_control
)[:, 1]

s_uplift = s_mu1 - s_mu0

print(
    pd.Series(
        s_uplift,
        name="s_uplift"
    ).describe()
)

count    150000.000000
mean          0.001073
std           0.010550
min          -0.389704
25%           0.000005
50%           0.000009
75%           0.000093
max           0.476671
Name: s_uplift, dtype: float64


# Parte G — T-Learner

In [15]:
treated_mask = T_train.eq(1)
control_mask = T_train.eq(0)

print("Treino tratado:", treated_mask.sum())
print("Treino controle:", control_mask.sum())

Treino tratado: 237927
Treino controle: 42073


In [16]:
t_model_treated = LGBMClassifier(
    **MODEL_PARAMS
)

t_model_control = LGBMClassifier(
    **MODEL_PARAMS
)

start = time.time()

t_model_treated.fit(
    X_train.loc[treated_mask],
    Y_train.loc[treated_mask]
)

t_model_control.fit(
    X_train.loc[control_mask],
    Y_train.loc[control_mask]
)

print(
    "Segundos:",
    round(time.time() - start, 1)
)

Segundos: 15.3


In [17]:
t_mu1 = t_model_treated.predict_proba(
    X_test
)[:, 1]

t_mu0 = t_model_control.predict_proba(
    X_test
)[:, 1]

t_uplift = t_mu1 - t_mu0

display(
    pd.Series(
        t_uplift,
        name="t_uplift"
    ).describe()
)

,t_uplift
count,150000.000000
mean,0.002230
std,0.023839
min,-0.711212
25%,0.000026
50%,0.000042
75%,0.000391
max,0.975402


# Parte H — funções de avaliação

In [18]:
def uplift_by_decile(
    treatment,
    outcome,
    uplift_score,
    n_bins=10
):
    table = pd.DataFrame({
        "treatment": np.asarray(treatment),
        "outcome": np.asarray(outcome),
        "uplift_score": np.asarray(uplift_score)
    })

    ranking = table["uplift_score"].rank(
        method="first",
        ascending=False
    )

    table["decile"] = pd.qcut(
        ranking,
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)

    rows = []

    for decile, group in table.groupby(
        "decile",
        sort=True
    ):
        treated = group[
            group["treatment"] == 1
        ]
        control = group[
            group["treatment"] == 0
        ]

        rows.append({
            "decile": decile,
            "customers": len(group),
            "predicted_uplift_mean":
                group["uplift_score"].mean(),
            "treated_rate":
                treated["outcome"].mean(),
            "control_rate":
                control["outcome"].mean(),
            "observed_uplift":
                treated["outcome"].mean()
                - control["outcome"].mean()
        })

    return pd.DataFrame(rows)

In [19]:
def qini_metrics(
    treatment,
    outcome,
    uplift_score
):
    table = pd.DataFrame({
        "treatment": np.asarray(treatment),
        "outcome": np.asarray(outcome),
        "uplift_score": np.asarray(uplift_score)
    }).sort_values(
        "uplift_score",
        ascending=False
    ).reset_index(drop=True)

    treated = table["treatment"].eq(1)
    control = table["treatment"].eq(0)

    cumulative_treated = treated.cumsum()
    cumulative_control = control.cumsum()

    cumulative_y_treated = (
        table["outcome"] * treated
    ).cumsum()

    cumulative_y_control = (
        table["outcome"] * control
    ).cumsum()

    safe_control = cumulative_control.replace(
        0,
        np.nan
    )

    incremental = (
        cumulative_y_treated
        - cumulative_y_control
        * cumulative_treated
        / safe_control
    ).fillna(0.0)

    population_share = (
        np.arange(1, len(table) + 1)
        / len(table)
    )

    random_line = (
        population_share
        * incremental.iloc[-1]
    )

    auuc = np.trapezoid(
        incremental,
        population_share
    )

    qini = np.trapezoid(
        incremental - random_line,
        population_share
    )

    curve = pd.DataFrame({
        "population_share": population_share,
        "incremental": incremental,
        "random": random_line
    })

    return {
        "auuc": float(auuc),
        "qini": float(qini),
        "curve": curve
    }

In [20]:
s_deciles = uplift_by_decile(
    T_test,
    Y_test,
    s_uplift
)

t_deciles = uplift_by_decile(
    T_test,
    Y_test,
    t_uplift
)

s_qini = qini_metrics(
    T_test,
    Y_test,
    s_uplift
)

t_qini = qini_metrics(
    T_test,
    Y_test,
    t_uplift
)

display(s_deciles)
display(t_deciles)

print("S Qini:", s_qini["qini"])
print("T Qini:", t_qini["qini"])

,decile,customers,predicted_uplift_mean,treated_rate,control_rate,observed_uplift
0,1,15000,0.011360,0.023379,0.014021,0.009358
1,2,15000,0.000484,0.001718,0.000912,0.000805
2,3,15000,0.000106,0.000785,0.000885,-0.000100
3,4,15000,0.000027,0.000236,0.000000,0.000236
4,5,15000,0.000012,0.000316,0.000000,0.000316
5,6,15000,0.000008,0.000000,0.000000,0.000000
6,7,15000,0.000006,0.000078,0.000000,0.000078
7,8,15000,0.000005,0.000158,0.000000,0.000158
8,9,15000,0.000002,0.000157,0.000000,0.000157
9,10,15000,-0.001279,0.003293,0.003560,-0.000267


,decile,customers,predicted_uplift_mean,treated_rate,control_rate,observed_uplift
0,1,15000,0.022598,0.024338,0.014385,0.009953
1,2,15000,0.001301,0.001950,0.001834,0.000116
2,3,15000,0.000408,0.000786,0.000440,0.000345
3,4,15000,0.000133,0.000315,0.000000,0.000315
4,5,15000,0.000054,0.000394,0.000000,0.000394
5,6,15000,0.000037,0.000237,0.000000,0.000237
6,7,15000,0.000030,0.000000,0.000000,0.000000
7,8,15000,0.000026,0.000079,0.000000,0.000079
8,9,15000,0.000020,0.000078,0.000000,0.000078
9,10,15000,-0.002307,0.001963,0.002647,-0.000683


S Qini: 66.00351174909183
T Qini: 71.3915358872895


# Parte I — DR-Learner com cross-fitting

In [21]:
X_cf = X_train.reset_index(drop=True)
T_cf = T_train.reset_index(drop=True)
Y_cf = Y_train.reset_index(drop=True)

mu1_oof = np.zeros(len(X_cf))
mu0_oof = np.zeros(len(X_cf))

kfold = KFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

In [22]:
for fold, (fit_idx, pred_idx) in enumerate(
    kfold.split(X_cf),
    start=1
):
    X_fit = X_cf.iloc[fit_idx]
    T_fit = T_cf.iloc[fit_idx]
    Y_fit = Y_cf.iloc[fit_idx]

    X_pred = X_cf.iloc[pred_idx]

    fold_treated = T_fit.eq(1)
    fold_control = T_fit.eq(0)

    model_1 = LGBMClassifier(
        **MODEL_PARAMS
    )
    model_0 = LGBMClassifier(
        **MODEL_PARAMS
    )

    model_1.fit(
        X_fit.loc[fold_treated],
        Y_fit.loc[fold_treated]
    )

    model_0.fit(
        X_fit.loc[fold_control],
        Y_fit.loc[fold_control]
    )

    mu1_oof[pred_idx] = (
        model_1.predict_proba(
            X_pred
        )[:, 1]
    )

    mu0_oof[pred_idx] = (
        model_0.predict_proba(
            X_pred
        )[:, 1]
    )

    print("Fold concluído:", fold)

Fold concluído: 1
Fold concluído: 2
Fold concluído: 3


In [24]:
e = float(T_cf.mean())

dr_pseudo_outcome = (
    mu1_oof
    - mu0_oof
    + T_cf.to_numpy()
    * (
        Y_cf.to_numpy() - mu1_oof
    )
    / e
    - (
        1 - T_cf.to_numpy()
    )
    * (
        Y_cf.to_numpy() - mu0_oof
    )
    / (1 - e)
)

display(
    pd.Series(
        dr_pseudo_outcome,
        name="dr_pseudo_outcome"
    ).describe()
)

,dr_pseudo_outcome
count,280000.000000
mean,0.001171
std,0.122430
min,-6.655078
25%,-0.000047
50%,-0.000007
75%,-0.000003
max,6.368636


In [25]:
dr_model = LGBMRegressor(
    n_estimators=250,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=100,
    subsample=0.80,
    colsample_bytree=0.80,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

dr_model.fit(
    X_cf,
    dr_pseudo_outcome
)

dr_uplift = dr_model.predict(
    X_test
)

print("DR-LEARNER TREINADO")

DR-LEARNER TREINADO


In [26]:
dr_deciles = uplift_by_decile(
    T_test,
    Y_test,
    dr_uplift
)

dr_qini = qini_metrics(
    T_test,
    Y_test,
    dr_uplift
)

display(dr_deciles)
print("DR Qini:", dr_qini["qini"])

,decile,customers,predicted_uplift_mean,treated_rate,control_rate,observed_uplift
0,1,15000,0.020308,0.017477,0.009667,0.007811
1,2,15000,0.000585,0.001636,0.001847,-0.000210
2,3,15000,0.000063,0.000858,0.001377,-0.000520
3,4,15000,0.000023,0.000553,0.000426,0.000128
4,5,15000,0.000005,0.000474,0.000000,0.000474
5,6,15000,-0.000003,0.000392,0.000000,0.000392
6,7,15000,-0.000003,0.000079,0.000000,0.000079
7,8,15000,-0.000003,0.000079,0.000000,0.000079
8,9,15000,-0.000003,0.000078,0.000000,0.000078
9,10,15000,-0.009953,0.008699,0.005357,0.003342


DR Qini: 24.509184149373045


# Parte J — valor de política duplamente robusto

In [27]:
def dr_policy_scores(
    observed_treatment,
    observed_outcome,
    recommended_treatment,
    mu1,
    mu0,
    propensity
):
    observed_treatment = np.asarray(
        observed_treatment
    )
    observed_outcome = np.asarray(
        observed_outcome
    )
    recommended_treatment = np.asarray(
        recommended_treatment
    )

    mu_recommended = np.where(
        recommended_treatment == 1,
        mu1,
        mu0
    )

    mu_observed = np.where(
        observed_treatment == 1,
        mu1,
        mu0
    )

    p_observed = np.where(
        observed_treatment == 1,
        propensity,
        1 - propensity
    )

    correction = (
        observed_treatment
        == recommended_treatment
    ) * (
        observed_outcome - mu_observed
    ) / p_observed

    return mu_recommended + correction

In [28]:
threshold = np.quantile(
    t_uplift,
    0.70
)

policy_treatment = (
    t_uplift >= threshold
).astype(int)

control_policy = np.zeros(
    len(T_test),
    dtype=int
)

policy_scores = dr_policy_scores(
    T_test,
    Y_test,
    policy_treatment,
    t_mu1,
    t_mu0,
    propensity=e
)

control_scores = dr_policy_scores(
    T_test,
    Y_test,
    control_policy,
    t_mu1,
    t_mu0,
    propensity=e
)

incremental_scores = (
    policy_scores - control_scores
)

print(
    "Valor incremental médio:",
    incremental_scores.mean()
)

Valor incremental médio: 0.0011312336424189173


In [29]:
def bootstrap_mean_ci(
    values,
    n_boot=500,
    seed=42
):
    values = np.asarray(values)
    rng_boot = np.random.default_rng(seed)
    estimates = np.empty(n_boot)

    for i in range(n_boot):
        sample_idx = rng_boot.integers(
            0,
            len(values),
            size=len(values)
        )
        estimates[i] = values[
            sample_idx
        ].mean()

    return {
        "estimate": float(values.mean()),
        "ci_low": float(
            np.quantile(estimates, 0.025)
        ),
        "ci_high": float(
            np.quantile(estimates, 0.975)
        )
    }

policy_ci = bootstrap_mean_ci(
    incremental_scores
)

display(policy_ci)

{'estimate': 0.0011312336424189173,
 'ci_low': 0.0005971305045999986,
 'ci_high': 0.0016758728356689198}

# Parte K — comparar e salvar

In [30]:
comparison = pd.DataFrame([
    {
        "model": "S-Learner",
        "qini": s_qini["qini"],
        "auuc": s_qini["auuc"],
        "top_decile_uplift":
            s_deciles.loc[
                s_deciles["decile"] == 1,
                "observed_uplift"
            ].iloc[0]
    },
    {
        "model": "T-Learner",
        "qini": t_qini["qini"],
        "auuc": t_qini["auuc"],
        "top_decile_uplift":
            t_deciles.loc[
                t_deciles["decile"] == 1,
                "observed_uplift"
            ].iloc[0]
    },
    {
        "model": "DR-Learner",
        "qini": dr_qini["qini"],
        "auuc": dr_qini["auuc"],
        "top_decile_uplift":
            dr_deciles.loc[
                dr_deciles["decile"] == 1,
                "observed_uplift"
            ].iloc[0]
    }
]).sort_values(
    "qini",
    ascending=False
)

display(comparison)

,model,qini,auuc,top_decile_uplift
1,T-Learner,71.391536,153.288914,0.009953
0,S-Learner,66.003512,147.900890,0.009358
2,DR-Learner,24.509184,106.406562,0.007811


In [31]:
RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)
MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

criteo_test_results = X_test.copy()
criteo_test_results["treatment"] = (
    T_test.values
)
criteo_test_results["conversion"] = (
    Y_test.values
)
criteo_test_results["response_score"] = (
    response_score
)
criteo_test_results["s_uplift"] = s_uplift
criteo_test_results["t_uplift"] = t_uplift
criteo_test_results["dr_uplift"] = dr_uplift

criteo_test_results.to_parquet(
    RESULTS_DIR
    / "criteo_test_predictions.parquet",
    index=False
)

comparison.to_csv(
    RESULTS_DIR
    / "criteo_model_comparison.csv",
    index=False
)

joblib.dump(
    t_model_treated,
    MODELS_DIR / "criteo_treated_model.joblib"
)
joblib.dump(
    t_model_control,
    MODELS_DIR / "criteo_control_model.joblib"
)
joblib.dump(
    dr_model,
    MODELS_DIR / "criteo_dr_model.joblib"
)

print("RESULTADOS SALVOS")

RESULTADOS SALVOS
